# Reto 6: Validador de Códigos con Expresiones Regulares

## Programación para Ciencia de Datos
### Instituto Politécnico Nacional - ESCOM
### 07 de Mayo

**Alumno:** Bryan Axel Esparrza Davila

---

In [1]:
import re
from typing import Dict, List
from datetime import date
import csv

# Departamentos válidos para empleados
DEPARTAMENTOS_VALIDOS = ['VEN', 'ADM', 'TEC', 'LOG', 'RHH']

# Series válidas para facturas
SERIES_VALIDAS = ['A', 'B', 'C', 'D', 'E']

In [2]:

# Funciones de validación individual

def validar_producto(codigo: str) -> Dict:
    resultado = {"valido": False, "categoria": None, "numero": None, "pais": None}
    
    # Explicación Regex: 
    # ^ y $ aseguran que no haya caracteres extra alrededor.
    # ([A-Z]{3}) captura exactamente 3 letras mayúsculas.
    # (\d{4}) captura exactamente 4 dígitos.
    # ([A-Z]{2}) captura exactamente 2 letras mayúsculas.
    patron = r'^([A-Z]{3})-(\d{4})-([A-Z]{2})$'
    match = re.match(patron, codigo)

    if match:
        resultado["valido"]    = True
        resultado["categoria"] = match.group(1)
        resultado["numero"]    = match.group(2)
        resultado["pais"]      = match.group(3)

    return resultado

def validar_envio(codigo: str) -> Dict:
    resultado = {"valido": False, "fecha": None, "secuencial": None}
    
    # Explicación Regex:
    # (\d{4}) captura el año.
    # (0[1-9]|1[0-2]) captura el mes asegurando formato 01-12.
    # (0[1-9]|[12]\d|3[01]) captura el día asegurando formato 01-31.
    # (\d{6}) captura 6 dígitos exactos para el secuencial.
    patron = r'^ENV-(\d{4})-(0[1-9]|1[0-2])-(0[1-9]|[12]\d|3[01])-(\d{6})$'
    match = re.match(patron, codigo)

    if match:
        anio = int(match.group(1))
        if 2020 <= anio <= 2030:
            resultado["valido"]      = True
            resultado["fecha"]       = f"{match.group(1)}-{match.group(2)}-{match.group(3)}"
            resultado["secuencial"]  = match.group(4)

    return resultado

def validar_empleado(codigo: str) -> Dict:
    resultado = {"valido": False, "departamento": None, "numero": None}
    
    # Explicación Regex:
    # ([A-Z]{3}) captura 3 letras para el departamento.
    # ([1-9]\d{3}) asegura 4 dígitos y que el primero no sea 0.
    patron = r'^EMP-([A-Z]{3})-([1-9]\d{3})$'
    match = re.match(patron, codigo)

    if match:
        depto = match.group(1)
        if depto in DEPARTAMENTOS_VALIDOS:
            resultado["valido"]       = True
            resultado["departamento"] = depto
            resultado["numero"]       = match.group(2)

    return resultado

def validar_factura(codigo: str) -> Dict:
    resultado = {"valido": False, "serie": None, "numero": None}
    
    # Explicación Regex:
    # ([A-E]) asegura que la serie sea exactamente una letra entre A y E.
    # (\d{6}) captura exactamente 6 dígitos.
    patron = r'^FAC-([A-E])-(\d{6})$'
    match = re.match(patron, codigo)

    if match:
        resultado["valido"]  = True
        resultado["serie"]   = match.group(1)
        resultado["numero"]  = match.group(2)

    return resultado

In [3]:
#Validador universal

def validar_codigo(codigo: str) -> Dict:
    resultado = {
        "codigo":   codigo,
        "tipo":     "desconocido",
        "valido":   False,
        "detalles": {}
    }

    mapa = {
        "ENV": ("envio",    validar_envio),
        "EMP": ("empleado", validar_empleado),
        "FAC": ("factura",  validar_factura),
    }

    prefijo = codigo[:3]

    if prefijo in mapa:
        tipo, fn = mapa[prefijo]
        resultado["tipo"] = tipo
        detalle = fn(codigo)
        resultado["valido"]   = detalle["valido"]
        resultado["detalles"] = detalle

    else:
        detalle = validar_producto(codigo)

        # ── AJUSTE PARA CLASIFICACIÓN DE PRODUCTOS ──
        partes = codigo.split("-")
        if detalle["valido"] or len(partes) == 3:
            resultado["tipo"]     = "producto"
            resultado["valido"]   = detalle["valido"]
            resultado["detalles"] = detalle

    return resultado

# Procesamiento por lotes


def procesar_lote(codigos: List[str]) -> Dict:
    resultado = {
        "total":    0,
        "validos":  0,
        "invalidos": 0,
        "por_tipo": {
            "producto":    {"total": 0, "validos": 0},
            "envio":       {"total": 0, "validos": 0},
            "empleado":    {"total": 0, "validos": 0},
            "factura":     {"total": 0, "validos": 0},
            "desconocido": {"total": 0, "validos": 0},
        },
        "detalle": []
    }

    for codigo in codigos:
        res = validar_codigo(codigo)
        tipo = res["tipo"]

        resultado["total"] += 1
        if res["valido"]:
            resultado["validos"] += 1
        else:
            resultado["invalidos"] += 1

        resultado["por_tipo"][tipo]["total"] += 1
        if res["valido"]:
            resultado["por_tipo"][tipo]["validos"] += 1

        resultado["detalle"].append(res)

    return resultado

# BONUS

def sugerir_correccion(codigo: str) -> str:
    sugerencia = codigo.strip().upper()
    res = validar_codigo(sugerencia)
    if res["valido"]:
        return f"¿Quisiste decir: {sugerencia}?"
    return "No se encontró una corrección automática."

def validar_fecha_real(anio: int, mes: int, dia: int) -> bool:
    try:
        date(anio, mes, dia)
        return True
    except ValueError:
        return False

def exportar_resultados(reporte: Dict, archivo: str) -> None:
    with open(archivo, "w", newline="", encoding="utf-8") as f:
        campos = ["codigo", "tipo", "valido"]
        writer = csv.DictWriter(f, fieldnames=campos, extrasaction="ignore")
        writer.writeheader()
        for fila in reporte["detalle"]:
            writer.writerow(fila)
    print(f"Resultados exportados a: {archivo}")

In [4]:
CODIGOS_PRUEBA = [
    "TEC-0001-MX", "ALI-9999-US", "ROB-1234-CA", "tec-0001-MX", "TEC-001-MX", "TECH-0001-MX",
    "ENV-2024-03-15-001234", "ENV-2025-12-01-999999", "ENV-2019-03-15-001234", "ENV-2024-13-15-001234", "ENV-2024-03-32-001234",
    "EMP-VEN-1234", "EMP-TEC-9999", "EMP-ADM-1000", "EMP-VEN-0123", "EMP-XXX-1234", "EMP-VEN-123",
    "FAC-A-123456", "FAC-E-000001", "FAC-B-999999", "FAC-F-123456", "FAC-A-12345", "FAC-a-123456",
    "XXX-1234", "RANDOM-CODE"
]

def mostrar_reporte(reporte: Dict) -> None:
    print("==============================================================")
    print("                 REPORTE DE VALIDACIÓN")
    print("==============================================================")
    print(f"\nTotal procesados: {reporte['total']}")
    
    if reporte['total'] > 0:
        print(f"Válidos: {reporte['validos']} ({reporte['validos']/reporte['total']*100:.1f}%)")
        print(f"Inválidos: {reporte['invalidos']} ({reporte['invalidos']/reporte['total']*100:.1f}%)")
    
    print("\nDesglose por tipo:")
    print("-" * 40)
    for tipo, stats in reporte["por_tipo"].items():
        if stats["total"] > 0:
            tasa = stats["validos"] / stats["total"] * 100
            print(f"  {tipo.capitalize():<12}:  {stats['validos']:>2}/ {stats['total']:<2} ({tasa:.0f}% válidos)")
    
    print("\n==============================================================")

# Ejecución
reporte_final = procesar_lote(CODIGOS_PRUEBA)
mostrar_reporte(reporte_final)
exportar_resultados(reporte_final, "resultados_validacion.csv")

                 REPORTE DE VALIDACIÓN

Total procesados: 25
Válidos: 11 (44.0%)
Inválidos: 14 (56.0%)

Desglose por tipo:
----------------------------------------
  Producto    :   3/ 6  (50% válidos)
  Envio       :   2/ 5  (40% válidos)
  Empleado    :   3/ 6  (50% válidos)
  Factura     :   3/ 6  (50% válidos)
  Desconocido :   0/ 2  (0% válidos)

Resultados exportados a: resultados_validacion.csv
